In [1]:
# svd_baseline.ipynb
%pip install -q transformers accelerate bitsandbytes torch sentence-transformers faiss-cpu pandas numpy seaborn scikit-learn implicit

# Импорт библиотек
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from collections import Counter
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import json
import re
from scipy.sparse import csr_matrix
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import normalize
import joblib
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

# Пути к файлам
DATA_PATH = "../data"
ratings = pd.read_csv(f"{DATA_PATH}/ratings.csv")
books = pd.read_csv(f"{DATA_PATH}/books.csv")
to_read = pd.read_csv(f"{DATA_PATH}/to_read.csv")
tags = pd.read_csv(f"{DATA_PATH}/tags.csv")
book_tags = pd.read_csv(f"{DATA_PATH}/book_tags.csv")

print("Данные загружены\n")

ARTIFACTS_PATH = "../artifacts/svd_models"
print(f"Путь для сохранения артефактов: {ARTIFACTS_PATH}")
os.makedirs(ARTIFACTS_PATH, exist_ok=True)

SRC_PATH = "../src/models"
print(f"Путь для сохранения лучшей модели: {SRC_PATH}")
os.makedirs(SRC_PATH, exist_ok=True)


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


c:\Users\andre\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Данные загружены

Путь для сохранения артефактов: ../artifacts/svd_models
Путь для сохранения лучшей модели: ../src/models


In [2]:
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd
import os
import pickle
import time

# --- Подготовка данных ---
# Создаем матрицу User-Item. 
user_ids = ratings['user_id'].unique()
book_ids = ratings['book_id'].unique()

user_id_map = {uid: i for i, uid in enumerate(user_ids)}
book_id_map = {bid: i for i, bid in enumerate(book_ids)}
user_id_inv = {i: uid for uid, i in user_id_map.items()}
book_id_inv = {i: bid for bid, i in book_id_map.items()}

ratings['user_idx'] = ratings['user_id'].map(user_id_map)
ratings['item_idx'] = ratings['book_id'].map(book_id_map)

# Формируем матрицу
csr_data = csr_matrix(
    (ratings['rating'].values.astype(float), (ratings['user_idx'].values, ratings['item_idx'].values)),
    shape=(len(user_ids), len(book_ids))
)

# Разделяем данные на Train и Test для оценки RMSE
train_ratings, test_ratings = train_test_split(ratings, test_size=0.2, random_state=42)

# Матрица для обучения (только train данные)
train_csr = csr_matrix(
    (train_ratings['rating'].values.astype(float), (train_ratings['user_idx'].values, train_ratings['item_idx'].values)),
    shape=(len(user_ids), len(book_ids))
)

# --- Параметры ---
param_grid = [
    # Базовые модели
    {'n_components': 32,  'n_iter': 20}
]

print(f"Всего конфигураций для обучения: {len(param_grid)}")

# --- Цикл обучения и оценки ---
results = []
best_rmse = float('inf')
best_model = None
best_params = None
best_user_factors = None
best_item_factors = None

K = 10  # Для совместимости структуры

for i, params in enumerate(param_grid):
    print(f"\n[Model {i+1}/{len(param_grid)}] Обучение с параметрами: {params}")
    start_time = time.time()
    
    # Инициализация модели
    svd_model = TruncatedSVD(
        n_components=params['n_components'],
        n_iter=params['n_iter'],
        random_state=42
    )
    
    # Обучение на train матрице
    svd_model.fit(train_csr)
    
    # Получение факторов
    # user_factors = U * Sigma 
    user_factors = svd_model.transform(train_csr)
    # item_factors = Vt
    item_factors = svd_model.components_.T
    
    # Оценка качества 
    test_users = test_ratings['user_idx'].values
    test_items = test_ratings['item_idx'].values
    true_ratings = test_ratings['rating'].values
    
    # Предсказание: dot product user vector и item vector
    predicted_ratings = np.sum(user_factors[test_users] * item_factors[test_items], axis=1)
    
    rmse = np.sqrt(mean_squared_error(true_ratings, predicted_ratings))
    elapsed_time = time.time() - start_time
    
    metrics = {
        'model_id': i + 1,
        'n_components': params['n_components'],
        'n_iter': params['n_iter'],
        'RMSE': rmse,
        'Time_sec': elapsed_time
    }
    results.append(metrics)
    
    print(f"RMSE: {rmse:.4f} | Time: {elapsed_time:.2f}s")
    
    # Сохранение лучшей модели
    if rmse < best_rmse:
        best_rmse = rmse
        best_model = svd_model
        best_params = params
        best_user_factors = user_factors
        best_item_factors = item_factors

# --- Вывод таблицы метрик ---
results_df = pd.DataFrame(results)
print("\n" + "="*90)
print("СВОДНАЯ ТАБЛИЦА МЕТРИК (SVD)")
print("="*90)
print(results_df.to_string(index=False))
print("="*90)

print(f"\nЛУЧШАЯ МОДЕЛЬ (по RMSE): Model с параметрами {best_params}")
print(f"Best RMSE: {best_rmse:.4f}")

# --- Сохранение лучшей модели и артефактов ---
print("\nСохранение артефактов лучшей модели...")

ARTIFACTS_PATH = "../artifacts/svd_models"
SRC_PATH = "../src/models"
os.makedirs(ARTIFACTS_PATH, exist_ok=True)
os.makedirs(SRC_PATH, exist_ok=True)

# Сохраняем факторы
np.save(os.path.join(SRC_PATH, "svd_user_factors.npy"), best_user_factors)
np.save(os.path.join(SRC_PATH, "svd_item_factors.npy"), best_item_factors)

# Сохраняем маппинги
with open(os.path.join(SRC_PATH, "svd_id_mappings.pkl"), 'wb') as f:
    pickle.dump({
        'user_id_map': user_id_map,
        'book_id_map': book_id_map,
        'user_id_inv': user_id_inv,
        'book_id_inv': book_id_inv
    }, f)

# Сохраняем сам объект SVD
joblib.dump(best_model, os.path.join(ARTIFACTS_PATH, "best_svd_model.joblib"))

print(f"Все модели сохранены в: {ARTIFACTS_PATH}")
print(f"Лучшая модель и маппинги сохранены в: {SRC_PATH}")

Всего конфигураций для обучения: 1

[Model 1/1] Обучение с параметрами: {'n_components': 32, 'n_iter': 20}
RMSE: 3.4982 | Time: 6.76s

СВОДНАЯ ТАБЛИЦА МЕТРИК (SVD)
 model_id  n_components  n_iter    RMSE  Time_sec
        1            32      20 3.49822  6.756254

ЛУЧШАЯ МОДЕЛЬ (по RMSE): Model с параметрами {'n_components': 32, 'n_iter': 20}
Best RMSE: 3.4982

Сохранение артефактов лучшей модели...
Все модели сохранены в: ../artifacts/svd_models
Лучшая модель и маппинги сохранены в: ../src/models
